# xbubekir-KI trainieren (Qwen3 1,7B → eigene xbubekir-KI im Browser)

**So geht's:** Oben im Menü *Laufzeit → Laufzeittyp ändern → **T4 GPU*** wählen und speichern. Dann die Zellen **der Reihe nach** mit ▶ starten.
Jede Zelle sagt am Ende, ob alles geklappt hat. Wenn etwas rot wird: Screenshot an Claude schicken.

**Einmalig vorher:** Links auf das Schlüssel-Symbol (*Secrets*) tippen → *Neues Secret* → Name `HF_TOKEN`, Wert = dein Hugging-Face-Token (mit Schreibrecht) → Schalter *Notebook-Zugriff* einschalten.

Dauer insgesamt etwa 45–60 Minuten. Grundmodell: Qwen3-1.7B (Apache-2.0).

## 1. Pakete und Trainingsdaten

In [ ]:
!pip -q uninstall -y torchao  # in Colab vorinstalliert, aber zu alt für peft (wird nicht gebraucht)
!pip -q install "transformers>=4.51,<5" "peft>=0.14" "accelerate>=1.2" huggingface_hub
import json, urllib.request
BRANCH = "main"  # aus diesem GitHub-Zweig werden die Trainingsdaten geladen
BASE = f"https://raw.githubusercontent.com/xbubekir68/xbubekir68.github.io/{BRANCH}/xbubekir/training/"
data = {}
for f in ["train.jsonl", "eval.jsonl"]:
    data[f] = [json.loads(l) for l in urllib.request.urlopen(BASE + f).read().decode("utf-8").splitlines() if l.strip()]
print("✅ Trainingsgespräche:", len(data["train.jsonl"]), "| Prüfgespräche:", len(data["eval.jsonl"]))

## 2. Grundmodell laden

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # weniger Speicherverschnitt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Keine GPU! Laufzeit → Laufzeittyp ändern → T4 GPU"
MODEL = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")  # halbe Größe, passt auf die T4
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("✅ Modell geladen:", MODEL)

## 3. Gespräche in Lernbeispiele umwandeln (gelernt wird nur, was xbubekir antwortet)

In [ ]:
MAXLEN = 1024
END = "<|im_end|>\n"
def examples(conv):
    msgs = conv["messages"]
    for j, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        prompt = tok.apply_chat_template(msgs[:j], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        p = tok(prompt, add_special_tokens=False)["input_ids"]
        a = tok(m["content"] + END, add_special_tokens=False)["input_ids"]
        ids = (p + a)[-MAXLEN:]
        labels = ([-100] * len(p) + a)[-MAXLEN:]
        yield {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}
train = [e for c in data["train.jsonl"] for e in examples(c)]
evals = [e for c in data["eval.jsonl"] for e in examples(c)]
x = train[0]
print("Beispiel – Eingabe:\n", tok.decode(x["input_ids"])[-400:])
print("\nBeispiel – davon gelernt:\n", tok.decode([t for t, l in zip(x["input_ids"], x["labels"]) if l != -100]))
print("\n✅ Lernbeispiele:", len(train), "| Prüfbeispiele:", len(evals))

## 4. Nachtrainieren (LoRA) – dauert etwa 20–30 Minuten

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
for p in model.parameters():  # nur die kleinen LoRA-Gewichte in voller Genauigkeit trainieren (nötig für fp16)
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()
args = TrainingArguments(output_dir="/content/run", num_train_epochs=2, per_device_train_batch_size=1,
    gradient_accumulation_steps=16, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.05,
    fp16=True, logging_steps=10, eval_strategy="epoch", save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evals,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
torch.cuda.empty_cache()
trainer.train()
print("✅ Training fertig. Prüf-Verlust:", round(trainer.evaluate()["eval_loss"], 3))

## 5. Probelauf: 10 Testfragen – bitte die Antworten anschauen

In [ ]:
model.eval()
model.config.use_cache = True
PERSONA = json.loads(urllib.request.urlopen(BASE + "persona.json").read().decode("utf-8"))["system"]
UEBUNG = next(c["messages"][0]["content"] for c in data["train.jsonl"] if "Übungspartner" in c["messages"][0]["content"])
tests = [(PERSONA, "Wer bist du?"), (PERSONA, "Kannst du im Internet suchen?"), (PERSONA, "Erklär mir kurz, was ein Verb ist."),
         (PERSONA, "Schreib eine kurze Nachricht an meinen Freund, dass ich später komme."), (PERSONA, "Was ist 7 mal 8?"),
         (UEBUNG, "مَرْحَبًا"), (UEBUNG, "Frag mich ab."), (UEBUNG, "Fang du an."),
         (UEBUNG, "Gib mir einen Satz."), (UEBUNG, "weiß ich nicht")]
for system, q in tests:
    text = tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=160, do_sample=False)
    print("🧑", q, "\n🤖", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip(), "\n" + "-" * 60)

## 6. Zusammenführen, speichern und auf Hugging Face sichern (privat)
Beim Start fragt Colab nach dem Zugriff auf `HF_TOKEN` → **„Zugriff gewähren“** antippen. Die Sicherung sorgt dafür, dass nicht neu trainiert werden muss, falls Colab die Sitzung beendet.

In [ ]:
merged = model.merge_and_unload().half()
merged.save_pretrained("/content/xbubekir-hf", safe_serialization=True)
tok.save_pretrained("/content/xbubekir-hf")
del model, merged, trainer
import gc; gc.collect(); torch.cuda.empty_cache()
# Sicherung: falls Colab die Sitzung beendet, holt Zelle 7 das trainierte Modell von hier zurück
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
BACKUP = api.whoami()["name"] + "/xbubekir-qwen3-1.7b-hf"
api.create_repo(BACKUP, private=True, exist_ok=True)
api.upload_folder(folder_path="/content/xbubekir-hf", repo_id=BACKUP)
print("✅ Gespeichert und privat gesichert: https://huggingface.co/" + BACKUP)


## 7. Ins Browser-Format umwandeln (WebLLM, 4-Bit) – dauert etwa 5–10 Minuten
Zuerst prüft ein Selbsttest den Umwandler am unveränderten Qwen3-1.7B gegen das offizielle WebLLM-Modell, dann wird dein Modell umgewandelt.

In [ ]:
import json, os, shutil, sys, urllib.request
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
if not os.path.exists("/content/xbubekir-hf/config.json"):  # Sitzung war weg: trainiertes Modell aus der Sicherung (Zelle 6) holen
    api = HfApi(token=userdata.get("HF_TOKEN"))
    snapshot_download(api.whoami()["name"] + "/xbubekir-qwen3-1.7b-hf", local_dir="/content/xbubekir-hf", token=api.token)
# Eigener Umwandler in reinem Python (ohne mlc_llm, das in Colab abstürzt), Aufbau 1:1 vom offiziellen WebLLM-Modell
urllib.request.urlretrieve("https://raw.githubusercontent.com/xbubekir68/xbubekir68.github.io/main/xbubekir/training/mlc_convert.py", "/content/mlc_convert.py")
sys.path.insert(0, "/content")
import importlib, mlc_convert; importlib.reload(mlc_convert)
OFFICIAL = "mlc-ai/Qwen3-1.7B-q4f16_1-MLC"
template = hf_hub_download(OFFICIAL, "tensor-cache.json")
# Selbsttest: das unveränderte Qwen3-1.7B umwandeln und mit dem offiziellen Modell vergleichen (Teil 3 enthält alle Gewichtsarten)
hf_hub_download("Qwen/Qwen3-1.7B", "model-00001-of-00002.safetensors", local_dir="/content/qwen3-base")
rounding = mlc_convert.selftest("/content/qwen3-base", template, {"params_shard_3.bin": hf_hub_download(OFFICIAL, "params_shard_3.bin")})
shutil.rmtree("/content/qwen3-base")
OUT = "/content/xbubekir-mlc"
shutil.rmtree(OUT, ignore_errors=True)  # Reste früherer Versuche entfernen
res = mlc_convert.convert("/content/xbubekir-hf", template, OUT, rounding, log=lambda m: print("\r" + m, end="", flush=True))
print()
official = {r["dataPath"]: r["md5sum"] for r in json.load(open(template))["records"]}
changed = sum(r["md5sum"] != official[r["dataPath"]] for r in res["records"])
assert changed >= len(res["records"]) // 2, "Fast alles gleich wie beim Grundmodell – ist das trainierte Modell in /content/xbubekir-hf? Screenshot an Claude"
# Konfiguration und Tokenizer vom offiziellen WebLLM-Modell übernehmen, damit es sicher zur fertigen Browser-Bibliothek passt
for f in ("mlc-chat-config.json", "tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt"):
    shutil.copy(hf_hub_download(OFFICIAL, f), f"{OUT}/{f}")
print(f"✅ Umgewandelt ({changed} von {len(res['records'])} Teilen weichen vom Grundmodell ab = nachtrainiert):", len(os.listdir(OUT)), "Dateien")

## 8. Auf Hugging Face hochladen

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi
api = HfApi(token=userdata.get("HF_TOKEN"))
user = api.whoami()["name"]
REPO = f"{user}/xbubekir-qwen3-1.7b-q4f16_1-MLC"
open("/content/xbubekir-mlc/README.md", "w").write(
    "---\nlicense: apache-2.0\nbase_model: Qwen/Qwen3-1.7B\n---\n# xbubekir (Qwen3-1.7B, nachtrainiert)\n\n"
    "Nachtrainierte Version von [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) (Apache-2.0) für die xbubekir-KI, "
    "im MLC-Format für WebLLM (q4f16_1).\n")
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="/content/xbubekir-mlc", repo_id=REPO)
print("✅ Fertig! Schick Claude diese Adresse:\nhttps://huggingface.co/" + REPO)